In [ ]:
import neo4j
import pandas as pd
from IPython.display import display
import numpy as np
import requests
from io import StringIO


In [ ]:
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2023/2023-08-22/population.csv"
response = requests.get(url, verify=False)
population = pd.read_csv(StringIO(response.text))
print("", population.shape)
population.head()

In [ ]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [ ]:
session = driver.session(database="neo4j")

In [ ]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [ ]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [ ]:
query = """
UNWIND range(1,10) AS id
CREATE (:Node {id:id})
"""

session.run(query)

In [ ]:
df = my_neo4j_run_query_pandas("""
MATCH (n:Node)
RETURN n.id
""")

display(df)

In [ ]:
# Step 1: Wipe the database first
my_neo4j_wipe_out_database()

# Step 2: Create COO (origin country) nodes
coo_rows = population[["coo_name"]].drop_duplicates().to_dict("records")
query_coo = """
UNWIND $rows AS row
MERGE (c:Country {name: row.coo_name, type: 'origin'})
"""
session.run(query_coo, rows=coo_rows)

# Step 3: Create COA (asylum country) nodes
coa_rows = population[["coa_name"]].drop_duplicates().to_dict("records")
query_coa = """
UNWIND $rows AS row
MERGE (c:Country {name: row.coa_name, type: 'asylum'})
"""
session.run(query_coa, rows=coa_rows)

# Step 4: Create Year nodes
year_rows = population[["year"]].drop_duplicates().to_dict("records")
query_year = """
UNWIND $rows AS row
MERGE (y:Year {year: row.year})
"""
session.run(query_year, rows=year_rows)

# Step 5: Create relationships with population/refugee data
# For example, COO -> COA for refugees
population_rows = population[["coo_name","coa_name","year","refugees","asylum_seekers"]].to_dict("records")

query_relationships = """
UNWIND $rows AS row
MATCH (origin:Country {name: row.coo_name, type:'origin'})
MATCH (asylum:Country {name: row.coa_name, type:'asylum'})
MATCH (y:Year {year: row.year})
MERGE (origin)-[r:REFUGEES_TO {refugees: row.refugees, asylum_seekers: row.asylum_seekers}]->(asylum)
MERGE (origin)-[:YEAR]->(y)
MERGE (asylum)-[:YEAR]->(y)
"""

session.run(query_relationships, rows=population_rows)

# Step 6: Check nodes
df_countries = my_neo4j_run_query_pandas("MATCH (c:Country) RETURN c.name AS country, c.type AS type LIMIT 10")
df_years = my_neo4j_run_query_pandas("MATCH (y:Year) RETURN y.year AS year LIMIT 10")

display(df_countries)
display(df_years)

In [ ]:
# Check GDS version
query = "RETURN gds.version() AS version"
df = my_neo4j_run_query_pandas(query)
display(df)

In [ ]:
query = """
MATCH (c:Country)-[r]-()
RETURN c.name AS country, COUNT(r) AS degree
ORDER BY degree DESC
LIMIT 10
"""
df_degree = my_neo4j_run_query_pandas(query)
display(df_degree)

In [ ]:
query = """
MATCH (origin:Country)-[r:REFUGEES_TO]->(asylum:Country)
RETURN origin.name AS origin, asylum.name AS asylum, r.refugees AS refugees
ORDER BY r.refugees DESC
LIMIT 10
"""
df_refugees = my_neo4j_run_query_pandas(query)
display(df_refugees)